In [81]:
using Graphs, Random, Base.Threads, GraphPlot, Colors, BenchmarkTools
using LinearAlgebra 
using SparseArrays  
using KrylovKit
using Plots
using Arpack
using DelimitedFiles
using IterativeSolvers
using Statistics, StatsBase, Distributions
using Printf
using CSV, DataFrames

# E-RAES

In this section we present an implementation in Julia language of the E-RAES model as described in [1].   

In [37]:
function e_raes_step_one!(g::SimpleGraph{Int}, d::Int; rng=Random.GLOBAL_RNG) 
    n = nv(g)
    new_edges = Tuple{Int, Int}[]
    deg_snapshot = degree(g)

    for u in 1:n
        need = max(0, d - deg_snapshot[u])
        need == 0 && continue

        forbidden = Set(neighbors(g, u))
        push!(forbidden, u)

        if length(forbidden) >= n 
            continue 
        end

        count = 0

        while count < need
            target = rand(rng, 1:n) 

            if target ∉ forbidden
                push!(new_edges, (u, target))
                #push!(forbidden, target) #including this line of code will allow us to emulate the dynamic without the replacement of the selected nodes
                count += 1
            end
        end
    end

    for (u, v) in new_edges
        if !has_edge(g, u, v) 
            add_edge!(g, u, v)
        end
    end
    return g
end


function e_raes_step_two!(g::SimpleGraph{Int}, d::Int, c::Real; rng=Random.GLOBAL_RNG) 
    deg_snapshot = degree(g)
    max_deg = ceil(Int, d * c)
    
    edges_to_remove = Set{Edge{Int}}()
    
    for u in vertices(g)
        excess = max(0, deg_snapshot[u] - max_deg)
        excess == 0 && continue
        
        current_neighbors = neighbors(g, u)
        to_drop = sample(rng, current_neighbors, excess, replace=true) # Setting replace=false will allow us to emulate the dynamic without the replacement of the selected nodes
        
        for v in to_drop
            push!(edges_to_remove, Edge(min(u,v), max(u,v)))
        end
    end
     
    for e in edges_to_remove
        rem_edge!(g, e)
    end
    
    return g
end

function e_raes_step_three!(g::SimpleGraph{Int}, p::Real; rng = Random.GLOBAL_RNG)
    p == 0 && return g
        for e in collect(edges(g))
            if rand(rng) < p
                rem_edge!(g, src(e), dst(e))
            end
        end

    return g
end


e_raes_step_three! (generic function with 1 method)

In [38]:
function spectral_gap(g::SimpleGraph{Int})
    if !is_connected(g) || nv(g) < 3
        return 0.0
    end

    degs = degree(g)
    if any(iszero, degs)
        return 0.0
    end

    n = nv(g)
    A = adjacency_matrix(g, Float64)
    D_inv_sqrt = 1.0 ./ sqrt.(degs)

    function L_mul(x)
        tmp = D_inv_sqrt .* x
        Ax = A * tmp
        return x .- D_inv_sqrt .* Ax
    end

    op = (x -> L_mul(x))
    vals, _, _ = eigsolve(op, n, 2, :SR; tol=1e-6, maxiter=500)

    return real(vals[2])
end

spectral_gap (generic function with 1 method)

In [39]:
function run_debug_experiment(params)
    local g = SimpleGraph(params.n)
    local rng = MersenneTwister(123) 

    println("Round | Archi Iniziali | Nodi < d (Pre 1) | Nodi < d (Post 1)| Nodi > c*d (Pre 2)| Archi Post-S1/2 | Nodi < d (Post 2) | Nodi > c*d (Post 2)| Connected |Spectral gap |Archi Finali")
    println("------|----------------|------------------|------------------|-------------------|-----------------|-------------------|--------------------|-----------|--------------|-------------")
    #println("Round | Archi Iniziali | Nodi < d (Pre 1) | Nodi < d (Post 1)| Nodi > c*d (Pre 2)| Archi Post-S1/2 | Nodi < d (Post 2) | Nodi > c*d (Post 2)| Connected | Archi Finali")
    #println("------|----------------|------------------|------------------|-------------------|-----------------|-------------------|--------------------|-----------|-------------")
    for r in 1:params.rounds
        archi_iniziali = ne(g)
        nodi_sotto_d_pre_s1 = count(<(params.d), degree(g))

        e_raes_step_one!(g, params.d; rng=rng)
        #DynamicRandomExpanderGenerator.check_degree(g, params.d)
        nodi_sotto_d_post_s1 = count(<(params.d), degree(g))
        nodi_sopra_cd_post_s1 = count(>(params.c * params.d), degree(g))
        e_raes_step_two!(g, params.d, params.c; rng=rng)
        deg = degree(g)
        avg_deg = mean(deg)
        #println("Round $r: Min Deg: $(minimum(deg)), Avg Deg: $avg_deg, Connected: $(is_connected(g))")
        archi_post_s12 = ne(g)
        nodi_sopra_cd_post_s2 = count(>(params.c * params.d), degree(g))
        nodi_sotto_d_post_s2 = count(<(params.d), degree(g))
        #println("Connected? ", is_connected(g))
        #println("Min degree: ", minimum(degree(g)))
        connected = is_connected(g)
        gap = spectral_gap(g)
        
        e_raes_step_three!(g, params.p; rng=rng)
        
        archi_finali = ne(g)
        
        @printf("%5d | %14d | %16d | %16d | %17d | %15d | %17d | %18d | %9s | %12f | %12d\n", r, archi_iniziali, nodi_sotto_d_pre_s1, nodi_sotto_d_post_s1, nodi_sopra_cd_post_s1, archi_post_s12, nodi_sotto_d_post_s2, nodi_sopra_cd_post_s2, string(connected), gap, archi_finali)
        #@printf("%5d | %14d | %16d | %16d | %17d | %15d | %17d | %18d | %9s | %12d\n", r, archi_iniziali, nodi_sotto_d_pre_s1, nodi_sotto_d_post_s1, nodi_sopra_cd_post_s1, archi_post_s12, nodi_sotto_d_post_s2, nodi_sopra_cd_post_s2, string(connected), archi_finali)
    end
end

run_debug_experiment (generic function with 1 method)

#### $n=2^{15}, d = 4, c=1.5,p=0.1$

In [40]:
PARAMS = (
    n = 2^15,
    d = 4,
    c = 1.5,
    p = 0.1,
    rounds = 15
)

run_debug_experiment(PARAMS)

Round | Archi Iniziali | Nodi < d (Pre 1) | Nodi < d (Post 1)| Nodi > c*d (Pre 2)| Archi Post-S1/2 | Nodi < d (Post 2) | Nodi > c*d (Post 2)| Connected |Spectral gap |Archi Finali
------|----------------|------------------|------------------|-------------------|-----------------|-------------------|--------------------|-----------|--------------|-------------
    1 |              0 |            32768 |                0 |             24889 |           77071 |              5029 |               1520 |     false |     0.000000 |        69210
    2 |          69210 |             9020 |                0 |              2720 |           78280 |              1275 |                 94 |      true |     0.187366 |        70506
    3 |          70506 |             6687 |                0 |              1223 |           77012 |               674 |                 18 |      true |     0.182481 |        69333
    4 |          69333 |             6724 |                0 |               964 |          

#### $n=2^{15}, d = 4, c=3,p=0.1$

In [41]:
PARAMS = (
    n = 2^15,
    d = 4,
    c = 3,
    p = 0.1,
    rounds = 15
)
run_debug_experiment(PARAMS)

Round | Archi Iniziali | Nodi < d (Pre 1) | Nodi < d (Post 1)| Nodi > c*d (Pre 2)| Archi Post-S1/2 | Nodi < d (Post 2) | Nodi > c*d (Post 2)| Connected |Spectral gap |Archi Finali
------|----------------|------------------|------------------|-------------------|-----------------|-------------------|--------------------|-----------|--------------|-------------
    1 |              0 |            32768 |                0 |               731 |          129925 |                10 |                 39 |      true |     0.338082 |       116895
    2 |         116895 |              539 |                0 |                11 |          117499 |                 1 |                  0 |      true |     0.308197 |       105655
    3 |         105655 |             1372 |                0 |                11 |          107221 |                 0 |                  0 |      true |     0.281238 |        96556
    4 |          96556 |             2268 |                0 |                 2 |          

#### $n=2^{15}, d = 4, c=1.5,p=1$

In [42]:
PARAMS = (
    n = 2^15,
    d = 4,
    c = 1.5,
    p = 1,
    rounds = 15
)

run_debug_experiment(PARAMS)

Round | Archi Iniziali | Nodi < d (Pre 1) | Nodi < d (Post 1)| Nodi > c*d (Pre 2)| Archi Post-S1/2 | Nodi < d (Post 2) | Nodi > c*d (Post 2)| Connected |Spectral gap |Archi Finali
------|----------------|------------------|------------------|-------------------|-----------------|-------------------|--------------------|-----------|--------------|-------------
    1 |              0 |            32768 |                0 |             24889 |           77071 |              5029 |               1520 |     false |     0.000000 |            0
    2 |              0 |            32768 |                0 |             24986 |           76910 |              5026 |               1460 |     false |     0.000000 |            0
    3 |              0 |            32768 |                0 |             25023 |           76991 |              4985 |               1496 |     false |     0.000000 |            0
    4 |              0 |            32768 |                0 |             24884 |          

#### $n=2^{15}, d = 4, c=2,p=1$

In [43]:
PARAMS = (
    n = 2^15,
    d = 4,
    c = 2,
    p = 1,
    rounds = 15
)

run_debug_experiment(PARAMS)

Round | Archi Iniziali | Nodi < d (Pre 1) | Nodi < d (Post 1)| Nodi > c*d (Pre 2)| Archi Post-S1/2 | Nodi < d (Post 2) | Nodi > c*d (Post 2)| Connected |Spectral gap |Archi Finali
------|----------------|------------------|------------------|-------------------|-----------------|-------------------|--------------------|-----------|--------------|-------------
    1 |              0 |            32768 |                0 |             12205 |          108253 |               534 |                871 |      true |     0.284418 |            0
    2 |              0 |            32768 |                0 |             12183 |          108525 |               493 |                858 |      true |     0.284293 |            0
    3 |              0 |            32768 |                0 |             12218 |          108426 |               520 |                896 |      true |     0.284476 |            0
    4 |              0 |            32768 |                0 |             12057 |          

#### $n=2^{15}, d = 4, c=2.5,p=1$

In [44]:
PARAMS = (
    n = 2^15,
    d = 4,
    c = 2.5,
    p = 1,
    rounds = 15
)

run_debug_experiment(PARAMS)

Round | Archi Iniziali | Nodi < d (Pre 1) | Nodi < d (Post 1)| Nodi > c*d (Pre 2)| Archi Post-S1/2 | Nodi < d (Post 2) | Nodi > c*d (Post 2)| Connected |Spectral gap |Archi Finali
------|----------------|------------------|------------------|-------------------|-----------------|-------------------|--------------------|-----------|--------------|-------------
    1 |              0 |            32768 |                0 |              3651 |          124901 |                99 |                250 |      true |     0.326635 |            0
    2 |              0 |            32768 |                0 |              3587 |          125044 |                91 |                266 |      true |     0.326591 |            0
    3 |              0 |            32768 |                0 |              3639 |          124991 |                93 |                246 |      true |     0.326231 |            0
    4 |              0 |            32768 |                0 |              3653 |          

#### $n=2^{15}, d = 4, c=3,p=1$

In [45]:
PARAMS = (
    n = 2^15,
    d = 4,
    c = 3,
    p = 1,
    rounds = 15
)

run_debug_experiment(PARAMS)

Round | Archi Iniziali | Nodi < d (Pre 1) | Nodi < d (Post 1)| Nodi > c*d (Pre 2)| Archi Post-S1/2 | Nodi < d (Post 2) | Nodi > c*d (Post 2)| Connected |Spectral gap |Archi Finali
------|----------------|------------------|------------------|-------------------|-----------------|-------------------|--------------------|-----------|--------------|-------------
    1 |              0 |            32768 |                0 |               731 |          129925 |                10 |                 39 |      true |     0.338082 |            0
    2 |              0 |            32768 |                0 |               684 |          129995 |                10 |                 35 |      true |     0.337472 |            0
    3 |              0 |            32768 |                0 |               683 |          130041 |                15 |                 39 |      true |     0.337417 |            0
    4 |              0 |            32768 |                0 |               685 |          

## Spectral Gap Comparison for ERAES

In this section we implement different methods to compute the spectral gap of the transition matrix of a simple random walk on a graph, and we compare them. The first two methods are based on the computation of the normalized Laplacian's eigenvalues, while the others explicitly compute $P=D^{-1}A.$ We recall that the spectral gap of a transition matrix $P=D^{-1}A$ for a simple random walk, namely $\gamma = 1 - \lambda_2(P),$ where $\lambda_2(P)$ is the second largest eigenvalue of $P,$ is equivalent to the second smallest eigenvalue of $L=I - D^{-1/2}AD^{-1/2},$ [2], [3]. 

In [46]:
function spectral_gap(g::SimpleGraph{Int})
    if !is_connected(g) || nv(g) < 3
        return 0.0
    end

    degs = degree(g)
    if any(iszero, degs)
        return 0.0
    end

    n = nv(g)
    A = adjacency_matrix(g, Float64)
    D_inv_sqrt = 1.0 ./ sqrt.(degs)

    function L_mul(x)
        tmp = D_inv_sqrt .* x
        Ax = A * tmp
        return x .- D_inv_sqrt .* Ax
    end

    op = (x -> L_mul(x))
    vals, _, _ = eigsolve(op, n, 2, :SR; tol=1e-6, maxiter=500)

    return real(vals[2])
end

function spectral_gap2(g::SimpleGraph{Int})

    if !is_connected(g) || nv(g) < 3
        return 0.0
    end
    degs = degree(g)
    if any(iszero, degs)
        return 0.0
    end

    A = adjacency_matrix(g, Float64)
    D_inv_sqrt = spdiagm(0 => 1.0 ./ sqrt.(degs))
    Id = spdiagm(0 => ones(Float64, nv(g)))
    L_norm = Id - D_inv_sqrt * A * D_inv_sqrt

    vals, _, _ = eigs(L_norm, nev=2, which=:SR)
    

    return real(vals[2])
end

function spectral_gap_classical(g::SimpleGraph{Int})
    if !is_connected(g) || nv(g) < 3; return 0.0; end
    degs = degree(g); if any(iszero, degs); return 0.0; end


    A = adjacency_matrix(g, Float64)
    D_inv = spdiagm(0 => 1.0 ./ degs)
    P = D_inv * A


    all_eigenvalues = eigvals(Matrix(P))
    
    sorted_vals = sort(real.(all_eigenvalues), rev=true)
    
    return sorted_vals[1] - sorted_vals[2]
end

function spectral_gap_paper_style(g::SimpleGraph{Int})

    if nv(g) <= 2 || !is_connected(g)
        return 0.0
    end

    degs = degree(g)
    if any(iszero, degs)
        return 0.0
    end

    A = adjacency_matrix(g, Float64)
    D_inv = spdiagm(0 => 1.0 ./ degs)
    P = D_inv * A


    vals, _, _ = eigsolve(P, 2, :LR)  
    vals = sort(real(vals); rev=true)
    λ1, λ2 = vals[1], vals[2]
    sg = λ1 - λ2  
    return sg
end


spectral_gap_paper_style (generic function with 1 method)

In [47]:
PARAMS = (n = 2^10, d = 4, c = 1.5, p = 0.1, rounds = 15)

function compare_methods()
    g = SimpleGraph(PARAMS.n)
    rng = MersenneTwister(123)
    skip = 0

    println("Round | Gap (KrylovKit)          |  Gap  (Arpack)             | Gap (Classical)             | Gap  (Paper style)          ")
    println("------|--------------------------|----------------------------|-----------------------------|-----------------------------")

    for r in 1:PARAMS.rounds
        e_raes_step_one!(g, PARAMS.d; rng=rng)
        e_raes_step_two!(g, PARAMS.d, PARAMS.c; rng=rng)

        gap1 = 0.0
        gap2 = 0.0
        gap3 = 0.0
        gap4 = 0.0


        if r >= skip
            gap1 = spectral_gap(g) # KrylovKit
            gap2 = spectral_gap2(g) # Arpack
            gap3 = spectral_gap_classical(g) # Classical
            gap4 = spectral_gap_paper_style(g) # Paper style
        end
        
        @printf("%5d | %24.6f | %26.6f | %26.6f  | %26.6f  \n", r, gap1, gap2, gap3, gap4)
        
        e_raes_step_three!(g, PARAMS.p; rng=rng)
    end
end

compare_methods()

Round | Gap (KrylovKit)          |  Gap  (Arpack)             | Gap (Classical)             | Gap  (Paper style)          
------|--------------------------|----------------------------|-----------------------------|-----------------------------
    1 |                 0.185315 |                   0.185315 |                   0.185315  |                   0.185315  
    2 |                 0.196138 |                   0.196138 |                   0.196138  |                   0.196138  
    3 |                 0.187572 |                   0.187572 |                   0.187572  |                   0.187572  
    4 |                 0.185243 |                   0.185243 |                   0.185243  |                   0.185243  
    5 |                 0.185466 |                   0.185466 |                   0.185466  |                   0.185466  
    6 |                 0.180656 |                   0.180656 |                   0.180656  |                   0.180656  
    7 |         

## Experiment 1 ERAES - Evolution of the spectral gap

In this section we try to replicate the experiment described in section $3.1$ of [1].
We analyze the evolution of the spectral gap for $n=2^{15}, d = 4$ and different values of $c,$ and we will compare them. We execute $100$ different independent simulations, each lasting $30$ rounds, to observe the stabilization of the spectral gap. 

The spectral gap is compute before the edge removal step.

In [48]:
function run_serial_spectral_gap_experiment(g::SimpleGraph, d::Int, c::Real, p::Real, rounds::Int; rng=Random.GLOBAL_RNG)
    spectral_gaps = Float64[]
    local skip_rounds = 0 # Set this variable to the number of initial rounds you wish to skip

    for r in 1:rounds
        e_raes_step_one!(g, d; rng=rng)
        e_raes_step_two!(g, d, c; rng=rng)

        local gap::Float64
        if r < skip_rounds
            gap = 0.0
        else
            gap = spectral_gap(g)
        end
        
        push!(spectral_gaps, gap)
        e_raes_step_three!(g, p; rng=rng)
    end
    return spectral_gaps
end

run_serial_spectral_gap_experiment (generic function with 1 method)

#### $n = 2^{15}, d = 4, c = 1.5$

In [ ]:
PARAMS = (
    n = 2^15,   # Numero di nodi 2^15
    d = 4,      # Grado minimo target
    c = 1.5,    # Fattore di grado massimo (max_degree = c * d)
    p = 0.1,    # Probabilità di fallimento degli archi
    rounds = 30 # Numero di round della simulazione
)
NUM_SIMULATIONS = 100 # 100

all_gaps = Matrix{Float64}(undef, PARAMS.rounds, NUM_SIMULATIONS)



Threads.@threads for i in 1:NUM_SIMULATIONS
    local g = SimpleGraph(PARAMS.n)
    local local_rng = MersenneTwister()
    println("Thread $(threadid()) avvia la simulazione $i.")

    all_gaps[:, i] = run_serial_spectral_gap_experiment(
            g, PARAMS.d, PARAMS.c, PARAMS.p, PARAMS.rounds;
            rng=local_rng   )
        
    println("Thread $(threadid()) ha completato la simulazione $i.")

end

In [49]:
df = CSV.read("../results/spectral_gaps_convergence_eraes_2.csv", DataFrame)
df

Row,Run,Round_1,Round_2,Round_3,Round_4,Round_5,Round_6,Round_7,Round_8,Round_9,Round_10,Round_11,Round_12,Round_13,Round_14,Round_15,Round_16,Round_17,Round_18,Round_19,Round_20,Round_21,Round_22,Round_23,Round_24,Round_25,Round_26,Round_27,Round_28,Round_29,Round_30
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,1.0,0.0,0.186474,0.182171,0.178444,0.177393,0.176455,0.175884,0.176526,0.175927,0.175441,0.175527,0.17552,0.175788,0.175539,0.175449,0.175693,0.175651,0.175216,0.175465,0.175436,0.174884,0.175648,0.175824,0.176169,0.17595,0.17608,0.175705,0.176365,0.175658,0.176291
2,2.0,0.0,0.187528,0.181586,0.178692,0.176798,0.176492,0.1756,0.175968,0.175596,0.175719,0.176308,0.175698,0.176227,0.175932,0.176213,0.175838,0.176405,0.175681,0.176032,0.176351,0.176512,0.175454,0.175446,0.175226,0.175836,0.175763,0.175704,0.176055,0.175759,0.175301
3,3.0,0.0,0.187089,0.182456,0.178721,0.176839,0.176393,0.175972,0.175588,0.176018,0.176109,0.17542,0.174682,0.175783,0.174807,0.175879,0.176224,0.175986,0.17616,0.176141,0.175772,0.175763,0.176116,0.176346,0.176638,0.175529,0.17627,0.176229,0.175597,0.175421,0.17573
4,4.0,0.0,0.186704,0.181885,0.178763,0.177861,0.176903,0.176039,0.175581,0.176064,0.175711,0.175912,0.176173,0.176,0.175772,0.17556,0.175767,0.175807,0.175509,0.174933,0.175285,0.175554,0.17582,0.176151,0.175638,0.175824,0.175377,0.175319,0.175599,0.175752,0.175568
5,5.0,0.0,0.187267,0.182754,0.178438,0.177316,0.176216,0.175122,0.175743,0.175639,0.175687,0.175776,0.175239,0.175755,0.175803,0.175965,0.175609,0.175523,0.175459,0.175608,0.176146,0.175984,0.175223,0.175808,0.175634,0.175314,0.175715,0.176007,0.175399,0.175556,0.175781
6,6.0,0.0,0.186234,0.182058,0.178542,0.176942,0.175711,0.175791,0.175737,0.176081,0.17587,0.176506,0.17604,0.175448,0.17514,0.175847,0.175882,0.175353,0.175504,0.175623,0.175495,0.175486,0.175908,0.175579,0.175854,0.175676,0.175608,0.175308,0.176123,0.176426,0.175978
7,7.0,0.0,0.186964,0.182344,0.178703,0.177497,0.176183,0.176226,0.17588,0.175907,0.175667,0.175772,0.176147,0.176391,0.176198,0.17594,0.175094,0.175849,0.175688,0.176275,0.176494,0.175726,0.175757,0.175467,0.176037,0.176129,0.175606,0.175894,0.175353,0.175249,0.175501
8,8.0,0.0,0.186877,0.182658,0.178176,0.1769,0.17681,0.175865,0.175568,0.175797,0.17671,0.176125,0.175845,0.175686,0.175824,0.176041,0.175534,0.175359,0.175449,0.175322,0.175403,0.176099,0.175913,0.175513,0.175692,0.175235,0.175829,0.176236,0.175854,0.176051,0.175777
9,9.0,0.0,0.187592,0.182015,0.178614,0.176703,0.176859,0.176574,0.17589,0.175963,0.175981,0.176168,0.176032,0.175811,0.175927,0.175617,0.175433,0.176064,0.175262,0.175275,0.175204,0.176091,0.17612,0.175952,0.175567,0.175789,0.176132,0.176357,0.176012,0.175398,0.175818


### Evolution of the spectral gap - c=1.5

![Evolution of the spectral gap - c=1.5](../results/eraes_classical/imgs/spectral_gaps_convergence_eraes_2.png)

#### $n = 2^{15}, d = 4, c = 3.0$

In [ ]:
PARAMS = (
    n = 2^15,   # Numero di nodi 2^15
    d = 4,      # Grado minimo target
    c = 3,    # Fattore di grado massimo (max_degree = c * d)
    p = 0.1,    # Probabilità di fallimento degli archi
    rounds = 30 # Numero di round della simulazione
)
NUM_SIMULATIONS = 100 # 100

all_gaps = Matrix{Float64}(undef, PARAMS.rounds, NUM_SIMULATIONS)



Threads.@threads for i in 1:NUM_SIMULATIONS
    local g = SimpleGraph(PARAMS.n)
    local local_rng = MersenneTwister()
    println("Thread $(threadid()) avvia la simulazione $i.")

    all_gaps[:, i] = run_serial_spectral_gap_experiment(
            g, PARAMS.d, PARAMS.c, PARAMS.p, PARAMS.rounds;
            rng=local_rng   )
        
    println("Thread $(threadid()) ha completato la simulazione $i.")

end

In [50]:
df = CSV.read("../results/spectral_gaps_convergence_eraes_main.csv", DataFrame)
df

Row,Run,Round_1,Round_2,Round_3,Round_4,Round_5,Round_6,Round_7,Round_8,Round_9,Round_10,Round_11,Round_12,Round_13,Round_14,Round_15,Round_16,Round_17,Round_18,Round_19,Round_20,Round_21,Round_22,Round_23,Round_24,Round_25,Round_26,Round_27,Round_28,Round_29,Round_30
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,1.0,0.337676,0.309245,0.282254,0.259293,0.239469,0.224727,0.213662,0.205216,0.199807,0.19567,0.192413,0.190646,0.187953,0.187603,0.187394,0.186757,0.186744,0.186383,0.186036,0.185603,0.185417,0.185602,0.18479,0.18482,0.184586,0.184747,0.185076,0.185212,0.185128,0.185307
2,2.0,0.337442,0.308577,0.282176,0.258541,0.23911,0.224182,0.212441,0.204268,0.198821,0.194949,0.191743,0.189958,0.188307,0.187002,0.186293,0.186053,0.185138,0.185794,0.186214,0.185156,0.184749,0.185577,0.185418,0.185178,0.185304,0.185129,0.185304,0.186165,0.185188,0.185277
3,3.0,0.337775,0.308619,0.281859,0.258489,0.239019,0.223836,0.211968,0.203963,0.198784,0.195351,0.192444,0.190178,0.188651,0.187734,0.186883,0.186636,0.186187,0.185654,0.185243,0.184939,0.185208,0.184782,0.184876,0.184525,0.184894,0.185403,0.185036,0.184998,0.184886,0.184803
4,4.0,0.337669,0.30854,0.28113,0.257898,0.239478,0.224252,0.21337,0.205191,0.199843,0.195435,0.192823,0.190276,0.189279,0.187574,0.187205,0.186737,0.186603,0.186333,0.18627,0.185982,0.185362,0.185061,0.184745,0.185057,0.184838,0.184864,0.185057,0.184989,0.184801,0.184751
5,5.0,0.33754,0.308622,0.281158,0.258227,0.239576,0.224405,0.21373,0.205474,0.199427,0.194983,0.191796,0.190105,0.1884,0.187458,0.186628,0.186466,0.18616,0.185894,0.185178,0.185805,0.185594,0.185006,0.184434,0.18526,0.184894,0.185412,0.185046,0.184771,0.185152,0.185132
6,6.0,0.337539,0.307755,0.281759,0.257795,0.238931,0.22404,0.2126,0.205012,0.198727,0.195719,0.192421,0.19054,0.187989,0.186435,0.186595,0.186501,0.185535,0.185095,0.185082,0.184981,0.184899,0.185277,0.185151,0.184464,0.184777,0.185057,0.184697,0.184886,0.185182,0.185472
7,7.0,0.337489,0.308478,0.282184,0.258026,0.239,0.224331,0.213758,0.205441,0.199823,0.195565,0.19285,0.190751,0.189194,0.187521,0.187235,0.186266,0.185834,0.1859,0.185649,0.185429,0.185253,0.185126,0.184247,0.185062,0.185762,0.185369,0.184732,0.184081,0.184683,0.185329
8,8.0,0.337293,0.308438,0.281135,0.257899,0.238956,0.223696,0.212373,0.204289,0.197525,0.194003,0.190995,0.189137,0.188381,0.18774,0.186549,0.186274,0.185164,0.185855,0.185947,0.185335,0.18513,0.184647,0.184398,0.185493,0.185202,0.185035,0.185052,0.184869,0.184939,0.184724
9,9.0,0.337366,0.309227,0.281996,0.25767,0.237841,0.223695,0.212437,0.203963,0.198446,0.19488,0.191962,0.190652,0.187774,0.187476,0.186737,0.185634,0.185291,0.185418,0.184884,0.185069,0.185146,0.184776,0.185245,0.185046,0.184493,0.184468,0.184797,0.184838,0.185295,0.184883


### Evolution of the spectral gap - c=3

![Evolution of the spectral gap - c=1.5](../results/eraes_classical/imgs/spectral_gaps_convergence_eraes_main.png)

## Experiment 2 ERAES - Average spectral gap in the long run

In this section, we replicate the experiment described in Section 3.2 of [1]. We simulate the E-RAES model and compute the average of the spectral gaps of the graph snapshots. The spectral gap is computed before the edge failure step.We propose two implementations:
- Fixed number of rounds to skip: Section 3.1 of [1] shows that, after about $15$ rounds, the spectral gap stabilizes. In this first version, we start to take into consideration the values of the spectral gap from that point.
- Dynamic number of rounds to skip: Section 3.1 of [1] introduces a heuristic criterion based on the stabilization of the spectral gap to determine when the evolution of the network reaches stationarity. We compute those values and use them as the number of rounds to skip.

In both implementations, we take into consideration the average over $100$ runs of $100$ rounds each.

### Fixed number of rounds to skip

In [ ]:
function calculate_average_gap_before_failure_eraes(
    n::Int, d::Int, c::Real, p::Real;
    num_runs::Int=10, # Number of independent simulations
    num_rounds::Int=100, # Rounds per simulation
    skip_initial_rounds::Int=30 # Rounds to discard for stabilization
)
    
    total_gaps_for_p = Float64[]


    for run in 1:num_runs
        g = SimpleGraph(n)
        rng = MersenneTwister(1234 + run + Int(p*1000) + n) 
        run_gaps = Float64[]

        for r in 1:num_rounds
            e_raes_step_one!(g, d; rng=rng)
            e_raes_step_two!(g, d, c; rng=rng)

            
            if r > skip_initial_rounds
                gap = spectral_gap(g)
                push!(run_gaps, gap)
            end
            
            
            e_raes_step_three!(g, p; rng=rng)
        end
        
        
        if !isempty(run_gaps)
            push!(total_gaps_for_p, mean(run_gaps)) 
        end
    end

 
    if isempty(total_gaps_for_p)
        @warn "Nessun gap stabile individuato per n=$n, p=$p"
        return 0.0
    end
    
    return mean(total_gaps_for_p)
end

In [ ]:
function run_single_task(task, idx)
    (n, p, i, j) = task
    local_result = calculate_average_gap_before_failure_eraes(
        n, D_PARAM, C_PARAM, p;
        num_runs=NUM_RUNS_PER_POINT,
        num_rounds=NUM_ROUNDS_PER_RUN,
        skip_initial_rounds=SKIP_ROUNDS
    )
    if idx % PRINT_EVERY == 0
        @info "Completati $idx / $num_tasks task"
    end

    return (i, j, local_result)
end
task_results = ThreadsX.map(run_single_task, tasks, 1:num_tasks)

for (i, j, value) in task_results
    results[i, j] = value
end

#### Results for c=3

In [ ]:
const N_VALUES = [2^9, 2^10, 2^11, 2^12, 2^13, 2^14, 2^15] # 512 to 32768 [2^9, 2^10, 2^11, 2^12, 2^13, 2^14, 2^15]
const P_VALUES = 0.0:0.1:1.0 # p from 0.0 to 1.0 in steps of 0.1
const D_PARAM = 4
const C_PARAM = 3.0
const NUM_RUNS_PER_POINT = 100 # Number of simulations to average for each (n, p) point 100
const NUM_ROUNDS_PER_RUN = 100 # Rounds per simulation run 100
const SKIP_ROUNDS = 15 # Initial rounds to discard


In [51]:
df = CSV.read("../results/average_spectral_gap_eraes_fixed_c3.csv", DataFrame)
df

Row,p_value,n=512,n=1024,n=2048,n=4096,n=8192,n=16384,n=32768
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,0.0,0.345852,0.342469,0.339885,0.339121,0.33825,0.337826,0.33758
2,0.1,0.192555,0.189118,0.187409,0.186272,0.185631,0.185275,0.185059
3,0.2,0.200884,0.197507,0.195585,0.194486,0.19383,0.193469,0.193241
4,0.3,0.211267,0.207698,0.205663,0.204563,0.203928,0.203536,0.203322
5,0.4,0.223489,0.219827,0.217731,0.216614,0.215942,0.215577,0.215375
6,0.5,0.237759,0.234057,0.23199,0.23083,0.230158,0.229773,0.22955
7,0.6,0.25453,0.250763,0.248599,0.24745,0.246759,0.246379,0.246126
8,0.7,0.273827,0.269981,0.267832,0.266669,0.265973,0.265561,0.265333
9,0.8,0.296027,0.292091,0.289919,0.288689,0.287995,0.287582,0.287354


![Spectral gap vs Edge failure rate fixed skip - c=3](../results/eraes_classical/imgs/average_spectral_gap_eraes_fixed_c3.png)

#### Results for c=1.5

In [ ]:
const N_VALUES = [2^9, 2^10, 2^11, 2^12, 2^13, 2^14, 2^15] # 512 to 32768 [2^9, 2^10, 2^11, 2^12, 2^13, 2^14, 2^15]
const P_VALUES = 0.0:0.1:1.0 # p from 0.0 to 1.0 in steps of 0.1
const D_PARAM = 4
const C_PARAM = 1.5
const NUM_RUNS_PER_POINT = 100 # Number of simulations to average for each (n, p) point 100
const NUM_ROUNDS_PER_RUN = 100 # Rounds per simulation run 100
const SKIP_ROUNDS = 15 # Initial rounds to discard


In [52]:
df = CSV.read("../results/average_spectral_gap_eraes_fixed_c15.csv", DataFrame)
df

Row,p_value,n=512,n=1024,n=2048,n=4096,n=8192,n=16384,n=32768
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,0.0,0.204534,0.200653,0.198722,0.197556,0.197133,0.19678,0.19654
2,0.1,0.183057,0.179853,0.178002,0.176977,0.176343,0.175982,0.175767
3,0.2,0.189347,0.186024,0.184124,0.183092,0.182419,0.181906,0.181755
4,0.3,0.196367,0.192857,0.190858,0.189696,0.188911,0.188369,0.187604
5,0.4,0.202924,0.199313,0.197262,0.195742,0.194076,0.192492,0.189369
6,0.5,0.207522,0.203545,0.200593,0.198116,0.193907,0.188296,0.173751
7,0.6,0.208237,0.203137,0.196998,0.190823,0.177274,0.155988,0.122999
8,0.7,0.202917,0.194427,0.18453,0.167912,0.137157,0.0976051,0.047257
9,0.8,0.192634,0.180549,0.160671,0.131302,0.0885388,0.0380136,0.0076073


![Spectral gap vs Edge failure rate fixed skip - c=1.5](../results/eraes_classical/imgs/average_spectral_gap_eraes_fixed_c15.png)

### Dynamic number of rounds to skip

As described in section 3.1 of [1], we compute, for each $(n,p),$ the value $\varepsilon$ as the mean absolute deviation of the non-zero values of the spectral gaps obtained from a pilot run (a pre-run) of $100$ rounds. 
On the same pre-run, we also determine the starting round $t_0$ in which the network becomes stable. Specifically, $t_0$ is defined as the first round for which all spectral gapss $\gamma_{t-\log n},\ldots, \gamma_{t}$ differs from $\gamma_t$ by at most $\varepsilon.$ 

In [ ]:
function compute_epsilon(spectral_gaps::Vector{Float64})
    non_zero_gaps = filter( x -> x > 0.0, spectral_gaps)

    if isempty(non_zero_gaps)
        return 0.0
    end

    mu = mean(non_zero_gaps)
    epsilon = mean(abs.(non_zero_gaps .- mu))
    return epsilon
end

function compute_t0(spectral_gaps::Vector{Float64}, n::Int, epsilon::Float64)
    window_size = ceil(Int, log2(n))
    
    if length(spectral_gaps) < window_size
        return -1
    end
    
    for i in window_size:length(spectral_gaps)
        start_index = i - window_size + 1
        window = @view spectral_gaps[start_index:i]
        
        gap_t = window[end] 
        

        is_stable = true
        for gap_i in window
            if abs(gap_i - gap_t) > epsilon
                is_stable = false
                break 
            end
        end

        if is_stable
            return i 
        end
    end
    
    return -1 
end

#### $\varepsilon$ values for $c=3$:

In [53]:
df = CSV.read("../results/stabilization_epsilon.csv", DataFrame)
df

Row,p_value,n=512,n=1024,n=2048,n=4096,n=8192,n=16384,n=32768
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,0.0,4.61292e-6,3.31975e-6,2.73972e-6,1.24754e-6,1.4742e-6,3.39024e-6,1.00395e-6
2,0.1,0.0106729,0.0118371,0.0111208,0.0109872,0.0109311,0.0109723,0.0109881
3,0.2,0.00815276,0.00602647,0.0060037,0.0058113,0.00572647,0.00572999,0.00572574
4,0.3,0.00646234,0.00484232,0.00421312,0.00377221,0.00374339,0.00380222,0.00375096
5,0.4,0.0059708,0.00421681,0.00334857,0.00302498,0.00282083,0.0027056,0.00269317
6,0.5,0.00569764,0.00420807,0.00311807,0.0027232,0.00241881,0.0022064,0.00220674
7,0.6,0.00559361,0.00432053,0.00319804,0.00273157,0.00222977,0.00200794,0.00196455
8,0.7,0.00607567,0.00411275,0.00292019,0.00212207,0.00195846,0.00166521,0.00156728
9,0.8,0.00549822,0.00318921,0.0025465,0.00208151,0.00167522,0.00123685,0.00112174


#### Stabilization round $t_0$ for $c=3.$

In [54]:
df = CSV.read("../results/stabilization_t0.csv", DataFrame)
df

Row,p_value,n=512,n=1024,n=2048,n=4096,n=8192,n=16384,n=32768
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,0.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0
2,0.1,16.0,20.0,21.0,21.0,22.0,23.0,24.0
3,0.2,53.0,22.0,19.0,17.0,19.0,19.0,20.0
4,0.3,23.0,13.0,21.0,15.0,16.0,17.0,18.0
5,0.4,11.0,14.0,33.0,14.0,15.0,16.0,17.0
6,0.5,31.0,-1.0,65.0,15.0,16.0,18.0,18.0
7,0.6,37.0,83.0,25.0,38.0,16.0,18.0,18.0
8,0.7,-1.0,-1.0,29.0,16.0,31.0,23.0,19.0
9,0.8,-1.0,-1.0,-1.0,-1.0,-1.0,21.0,19.0


#### $\varepsilon$ values for $c=1.5$:

In [82]:
df = CSV.read("../results/eraes_classical/stabilization_epsilon_c15.csv", DataFrame)
df

Row,p_value,n=512,n=1024,n=2048,n=4096,n=8192,n=16384,n=32768
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,0.0,9.84503e-5,0.000343558,0.000247096,5.29651e-5,8.60236e-5,6.688e-5,7.15427e-5
2,0.1,0.0040077,0.00289339,0.00169539,0.0011813,0.000923651,0.000609287,0.00051499
3,0.2,0.00462861,0.00261087,0.00169543,0.00108827,0.000700043,0.000515729,0.000312739
4,0.3,0.00417978,0.00266115,0.00172126,0.00110302,0.000788095,0.000481041,0.000347667
5,0.4,0.00418887,0.00291441,0.00164027,0.00105304,0.000683783,0.000456637,0.000341031
6,0.5,0.0042604,0.00281484,0.00171773,0.00151771,0.00102977,0.00108057,0.000535745
7,0.6,0.00402678,0.0032442,0.00184647,0.0017062,0.000952511,0.0017737,0.00344675
8,0.7,0.00438873,0.00250462,0.00230941,0.00240645,0.00137582,0.00269923,0.00118271
9,0.8,0.00589967,0.00370031,0.00240204,0.00204196,0.00372945,0.000952032,0.00451195


#### Stabilization round $t_0$ for $c=1.5$

In [83]:
df = CSV.read("../results/eraes_classical/stabilization_t0_c15.csv", DataFrame)
df

Row,p_value,n=512,n=1024,n=2048,n=4096,n=8192,n=16384,n=32768
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,0.0,12.0,13.0,15.0,16.0,17.0,17.0,18.0
2,0.1,-1.0,-1.0,-1.0,-1.0,61.0,-1.0,27.0
3,0.2,15.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
4,0.3,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,45.0
5,0.4,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
6,0.5,-1.0,64.0,-1.0,-1.0,-1.0,23.0,-1.0
7,0.6,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
8,0.7,99.0,24.0,-1.0,-1.0,-1.0,-1.0,-1.0
9,0.8,-1.0,59.0,-1.0,-1.0,-1.0,-1.0,15.0


In [ ]:
function calculate_average_gap_before_failure_eraes(
    n::Int, d::Int, c::Real, p::Real;
    num_runs::Int=10, # Number of independent simulations
    num_rounds::Int=100, # Rounds per simulation
    skip_initial_rounds::Int=30 # Rounds to discard for stabilization
)
    
    total_gaps_for_p = Float64[]


    for run in 1:num_runs
        g = SimpleGraph(n)
        rng = MersenneTwister(1234 + run + Int(p*1000) + n) 
        run_gaps = Float64[]

        for r in 1:num_rounds

            e_raes_step_one!(g, d; rng=rng)
            e_raes_step_two!(g, d, c; rng=rng)


            if r > skip_initial_rounds
                gap = spectral_gap(g)
                push!(run_gaps, gap)
            end
            

            e_raes_step_three!(g, p; rng=rng)
        end
        

        if !isempty(run_gaps)
            push!(total_gaps_for_p, mean(run_gaps)) 
        end
    end


    if isempty(total_gaps_for_p)
        @warn "No stable gaps collected for n=$n, p=$p"
        return 0.0
    end
    
    return mean(total_gaps_for_p)
end

We notice that, for certain values of $(n,p)$, the simulations are not able to find a round in which the network becomes stable. To determine the number of rounds to skip, we reason as follows:
- If $p \geq 0.8$: The graph is destroyed and re-built almost entirely at each round. There is not a long 'transition phase' until stationarity is reached, so we skip $\log_2(n)$ rounds, which is the minimum length of the window required by the stability criterion.
- If $local\_t0 = -1$, then a stable round was not found: This failure occurs predominantly for medium-to-high values of $p$ ($p \ge 0.5$). In these cases, the network is highly chaotic, and the graph snapshots vary too much to fit the stringent stability window ($\log_2 n$) within the allotted 100 rounds. The pilot run fails because the mixing time is longer than the pilot duration, or the fluctuations are simply too large. In this scenario, we default to the empirical evidence from Section 3.1 of [1] and take a fixed number of rounds ($15$) to skip.
- In the last case ($p < 0.8$ and $local\_t0 \ne -1$): A stable round was successfully found. We take $local\_t0$ as the number of rounds to skip.

In [ ]:
function run_single_task(task, idx)
    (n, p, i, j) = task
    

    local_t0 = t0_matrix[i, j]
    min_skip = ceil(Int, log2(n)) 
    
    local skip_rounds_dynamic::Int
    
    if p >= 0.8 
        skip_rounds_dynamic = min_skip
    elseif local_t0 == -1 
        skip_rounds_dynamic = FALLBACK_SKIP_ROUNDS
    else 
        skip_rounds_dynamic = local_t0
    end


    local_result = DynamicRandomExpanderGenerator.calculate_average_gap_before_failure_eraes(
        n, D_PARAM, C_PARAM, p;
        num_runs=NUM_RUNS_PER_POINT,
        num_rounds=NUM_ROUNDS_PER_RUN,
        skip_initial_rounds=skip_rounds_dynamic 
    )
    
    if idx % 1 == 0 
        @info "Completato $idx/$num_tasks: (n=$n, p=$p) -> (t0 usato: $skip_rounds_dynamic) -> Risultato: $(round(local_result, digits=4))"
    end

    return (i, j, local_result)
end

#### Results for c = 3

In [ ]:
const N_VALUES = [2^9, 2^10, 2^11, 2^12, 2^13, 2^14, 2^15]
const P_VALUES = 0.0:0.1:1.0
const D_PARAM = 4
const C_PARAM = 3.0
const NUM_RUNS_PER_POINT = 100
const NUM_ROUNDS_PER_RUN = 100
const FALLBACK_SKIP_ROUNDS = 15 

In [ ]:
df = CSV.read("../results/average_spectral_gap_eraes_main.csv", DataFrame)
df

### Spectral gap vs Edge failure rate dynamic - c=3

![Evolution of the spectral gap dynamic - c=3](../results/eraes_classical/imgs/average_spectral_gap_eraes_main.png)

#### Results for c = 1.5

In [ ]:
const N_VALUES = [2^9, 2^10, 2^11, 2^12, 2^13, 2^14, 2^15]
const P_VALUES = 0.0:0.1:1.0
const D_PARAM = 4
const C_PARAM = 1.5 
const NUM_RUNS_PER_POINT = 100
const NUM_ROUNDS_PER_RUN = 100
const FALLBACK_SKIP_ROUNDS = 15 

In [ ]:
df = CSV.read("../results/average_spectral_gap_eraes_main_small_c.csv", DataFrame)
df

### Spectral gap vs Edge failure rate dynamic - c=1.5

![Evolution of the spectral gap dynamic - c=1.5](../results/eraes_classical/imgs/average_spectral_gap_eraes_main_small_c.png)

## Flooding ERAES

In this section we replicate the flooding experiment according to section $3.3$ in [1].

In [ ]:
function flooding_step!(all_informed::BitVector, g::SimpleGraph, active_nodes::BitVector)

    all_informed .&= active_nodes

    newly_informed = Set{Int}()

    for u in findall(all_informed)

        for v in neighbors(g, u)

            if v <= length(active_nodes) && active_nodes[v] && !all_informed[v]
                push!(newly_informed, v)
            end
        end
    end


    for v in newly_informed
        if v <= length(all_informed)
            all_informed[v] = true
        end
    end

    return nothing
end

In [ ]:
PARAMS = (n = 2^15, d = 4, c = 1.5, p = 0.9, rounds = 50, q = 0.1)

function test_eraes_flooding(n::Int, d::Int, c::Real, p::Real, rounds::Int; seed = 1)
    println("\ Avvio Test Flooding E-RAES ")
    println("Parametri: n=$n, d=$d, c=$c, p=$p, rounds=$rounds")
    
    local g = SimpleGraph(n)
    local rng = MersenneTwister(seed)


    local active_nodes = trues(n) 
    local all_informed = falses(n)


    local starter_node = rand(rng, 1:n)
    all_informed[starter_node] = true
    @printf("Round 0: Nodo iniziale %d. Informati: 1 / %d\n", starter_node, n)


    for r in 1:rounds
        
  
        e_raes_step_one!(g, d; rng=rng)
        e_raes_step_two!(g, d, c; rng=rng)
        e_raes_step_three!(g, p; rng=rng)


        flooding_step!(all_informed, g, active_nodes)


        current_informed_count = count(all_informed)
        @printf("Round %d: Informati %d / %d nodi\n", r, current_informed_count, n)
        if r < 6
            println("\n   [Debug Dettagliato Round $r]")
            
            informed_list = findall(all_informed)
            sort!(informed_list) 
            
            println("   Lista Nodi Informati: ", informed_list)
            println("   Dettaglio Vicinato (sul grafo attuale):")
            
            for u in informed_list

                current_neighbors = sort(collect(neighbors(g, u)))
                @printf("     - Nodo %d -> Vicini: %s\n", u, current_neighbors)
            end
            println("   [Fine Debug Dettagliato Round $r]")
        end


        if current_informed_count == n
            println("Flooding E-RAES Completo")
            break
        end
    end
end

test_eraes_flooding(PARAMS.n, PARAMS.d, PARAMS.c, PARAMS.p, PARAMS.rounds)

#### c=3

In [ ]:
N_VALUES = [2^9, 2^10, 2^11, 2^12, 2^13, 2^14, 2^15]
P_VALUES = 0.0:0.1:0.9
D_PARAM = 4
C_PARAM = 3.0 
NUM_RUNS_PER_POINT = 100
NUM_ROUNDS_PER_RUN = 100
FALLBACK_SKIP_ROUNDS = 15 

In [86]:
df = CSV.read("../results/eraes_classical/average_flooding_time_eraes_main.csv", DataFrame)
df

Row,p_value,n=512,n=1024,n=2048,n=4096,n=8192,n=16384,n=32768
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,0.0,4.73,5.0,5.25,5.98,6.02,6.6,7.0
2,0.1,6.59,7.13,7.94,8.3,9.07,9.82,10.18
3,0.2,6.95,7.6,8.29,8.99,9.61,10.13,10.92
4,0.3,7.36,8.07,8.72,9.5,10.15,10.96,11.48
5,0.4,7.92,8.61,9.41,10.21,10.95,11.71,12.39
6,0.5,8.51,9.35,10.25,11.07,11.88,12.59,13.45
7,0.6,9.55,10.54,11.38,12.31,13.26,14.21,15.27
8,0.7,11.09,12.09,13.31,14.5,15.52,16.56,17.54
9,0.8,14.1,15.56,17.38,18.21,19.93,21.01,22.59


![Average flooding time - c = 3](../results/eraes_classical/imgs/average_flooding_time_eraes_main.png)

#### c=1.5

In [87]:
df = CSV.read("../results/eraes_classical/average_flooding_time_eraes_c15.csv", DataFrame)
df

Row,p_value,n=512,n=1024,n=2048,n=4096,n=8192,n=16384,n=32768
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,0.0,6.0,6.78,7.08,7.99,8.15,9.0,9.34
2,0.1,6.81,7.28,8.02,8.86,9.28,10.0,10.67
3,0.2,7.12,7.88,8.44,9.1,9.91,10.54,11.18
4,0.3,7.51,8.26,9.11,9.84,10.47,11.23,11.97
5,0.4,8.16,9.0,9.89,10.53,11.37,12.18,12.89
6,0.5,9.05,9.95,10.91,11.88,12.59,13.54,14.48
7,0.6,10.42,11.38,12.58,13.6,14.47,15.72,16.74
8,0.7,12.98,14.2,15.53,16.73,18.06,19.43,20.64
9,0.8,17.97,19.29,20.83,23.13,25.16,26.89,28.72


![Average flooding time - c = 1.5](../results/eraes_classical/imgs/average_flooding_time_eraes_c15.png)

# V-RAES

In [55]:
mutable struct vraes_graph
    g::SimpleGraph{Int}
    active_nodes::BitVector  # active_nodes[i]=true se nodo available, false altrimenti
    informed_nodes::BitVector  # Per flooding, informed_nodes[i] = true se nodo i è informato
    rng::AbstractRNG
    
    function vraes_graph(n::Int, seed::Int)
        new(
            SimpleGraph(n),       # g
            trues(n),          # active_nodes 
            falses(n),          # informed_nodes 
            MersenneTwister(seed) # rng
        )
    end
end

function v_raes_step_zero!(state::vraes_graph, lambda::Float64)
    n_old = nv(state.g)
    num_nuovi_nodi = rand(state.rng, Poisson(lambda))
    if num_nuovi_nodi == 0
        return n_old
    end
    
    add_vertices!(state.g, num_nuovi_nodi)

    append!(state.active_nodes, trues(num_nuovi_nodi))
    append!(state.informed_nodes, falses(num_nuovi_nodi)) 
    
    return n_old
end

function v_raes_step_one!(state::vraes_graph, d::Int, n_old::Int)
    g = state.g

    n_total = nv(g) 
    new_edges = Tuple{Int, Int}[]
    deg_snapshot = degree(g)

    for u in 1:n_total

        if !state.active_nodes[u]
            continue
        end
        
        need = max(0, d - deg_snapshot[u])
        need == 0 && continue

        forbidden = Set(neighbors(g, u))
        push!(forbidden, u)

        count = 0

        attempts = 0
        max_attempts = need * 50 

        while count < need && attempts < max_attempts
            attempts += 1
            

            target = rand(state.rng, 1:n_old)
            if state.active_nodes[target] && target ∉ forbidden
                push!(new_edges, (u, target))
                #push!(forbidden, target) without replacement
                count += 1
            end
        end
    end
    
    for (u, v) in new_edges
        !has_edge(g, u, v) && add_edge!(g, u, v)
    end
end

function v_raes_step_two!(state::vraes_graph, d::Int, c::Real)
    g = state.g
    rng = state.rng
    max_deg = ceil(Int, d * c)


    edges_to_remove = Set{Edge{Int}}() 
    

    deg_snapshot = degree(g)


    for u in findall(state.active_nodes)
        
        deg_u = deg_snapshot[u] 
        excess = max(0, deg_u - max_deg)
        
        excess == 0 && continue
        
        current_neighbors = neighbors(g, u)
        
        neighbors_to_drop = sample(rng, current_neighbors, excess, replace=true) #if replace=false then the algorithm will be without replacement
        
        for v in neighbors_to_drop
            push!(edges_to_remove, Edge(u, v)) 
        end
    end
    
    for e in edges_to_remove
        rem_edge!(g, e)
    end
end

function v_raes_step_three!(state::vraes_graph, q::Float64)
    g = state.g
    nodes_to_remove = Int[]
    
    for u in findall(state.active_nodes)
        if rand(state.rng) < q
            push!(nodes_to_remove, u)
        end
    end

    if isempty(nodes_to_remove)
        return
    end

    for u in nodes_to_remove
        state.active_nodes[u] = false
        state.informed_nodes[u] = false

        neighbors_copy = collect(neighbors(g, u))
        for v in neighbors_copy
            rem_edge!(g, u, v)
        end
    end
end

v_raes_step_three! (generic function with 1 method)

In [56]:
function count_active_edges(g::SimpleGraph, active::BitVector)
    s = 0
    for u in vertices(g)
        if active[u]
            for v in neighbors(g, u)
                if active[v] && u < v  
                    s += 1
                end
            end
        end
    end
    return s
end

function run_debug_experiment(params)

    state = vraes_graph(params.n, 123)
    λ = Float64(params.q * params.n)


    

    println("Round | Nodi Attivi | Nodi Post step 0 | Archi Attivi | Nodi < d (Pre 1) | Nodi < d (Post 1) | Archi Post-S1 | Nodi > c*d (Pre 2) | Archi Post-S2 | Nodi < d (Post 2) | Nodi > c*d (Post 2) | Nodi Post 3 | Archi Post 3")
    println("------|-------------|------------------|--------------|------------------|-------------------|---------------|--------------------|---------------|-------------------|---------------------|-------------|-------------")

    for r in 1:params.rounds

        nodi_attivi = count(state.active_nodes)
        archi_attivi = count_active_edges(state.g, state.active_nodes)

  
        n_old = v_raes_step_zero!(state, λ)
        nodi_post_step0 = count(state.active_nodes)

 
        nodi_sotto_d_pre_s1 = count(<(params.d), [degree(state.g, u) for u in vertices(state.g) if state.active_nodes[u]])
        v_raes_step_one!(state, params.d, n_old)
        nodi_sotto_d_post_s1 = count(<(params.d), [degree(state.g, u) for u in vertices(state.g) if state.active_nodes[u]])
        nodi_sopra_cd_post_s1 = count(>(params.c*params.d), [degree(state.g, u) for u in vertices(state.g) if state.active_nodes[u]])
        archi_post_s1 = count_active_edges(state.g, state.active_nodes)

 
        v_raes_step_two!(state, params.d, params.c)
        archi_post_s2 = count_active_edges(state.g, state.active_nodes)
        nodi_sopra_cd_post_s2 = count(>(params.c*params.d), [degree(state.g, u) for u in vertices(state.g) if state.active_nodes[u]])
        nodi_sotto_d_post_s2 = count(<(params.d), [degree(state.g, u) for u in vertices(state.g) if state.active_nodes[u]])

 
        v_raes_step_three!(state, params.q)
        nodi_post_s3 = count(state.active_nodes)
        archi_post_s3 = count_active_edges(state.g, state.active_nodes)

      
        @printf("%5d | %11d | %16d | %12d | %16d | %17d | %13d | %18d | %13d | %17d | %19d | %11d | %12d \n",
                r, nodi_attivi, nodi_post_step0, archi_attivi,
                nodi_sotto_d_pre_s1, nodi_sotto_d_post_s1, archi_post_s1,
                nodi_sopra_cd_post_s1, archi_post_s2, nodi_sotto_d_post_s2,
                nodi_sopra_cd_post_s2, nodi_post_s3, archi_post_s3)
    end
end

run_debug_experiment (generic function with 1 method)

In [58]:
PARAMS = (
    n = 2^15,
    d = 4,
    c = 1.5,
    q = 0.5,
    rounds = 30
)

(n = 32768, d = 4, c = 1.5, q = 0.5, rounds = 30)

In [59]:
run_debug_experiment(PARAMS)

Round | Nodi Attivi | Nodi Post step 0 | Archi Attivi | Nodi < d (Pre 1) | Nodi < d (Post 1) | Archi Post-S1 | Nodi > c*d (Pre 2) | Archi Post-S2 | Nodi < d (Post 2) | Nodi > c*d (Post 2) | Nodi Post 3 | Archi Post 3
------|-------------|------------------|--------------|------------------|-------------------|---------------|--------------------|---------------|-------------------|---------------------|-------------|-------------
    1 |       32768 |            49193 |            0 |            49193 |                 5 |        196752 |              30757 |        101052 |             18314 |                3337 |       24659 |        25587 
    2 |       24659 |            40981 |        25587 |            37605 |                 4 |        139573 |              20806 |         86197 |             14394 |                2514 |       20319 |        21092 
    3 |       20319 |            36738 |        21092 |            33971 |                 5 |        126808 |              18207 

# EV-RAES

In [60]:
mutable struct evraes_graph
    g::SimpleGraph{Int}
    active_nodes::BitVector  # active_nodes[i]=true se nodo available, false altrimenti
    informed_nodes::BitVector  # Per flooding, informed_nodes[i] = true se nodo i è informato
    rng::AbstractRNG
    
    function evraes_graph(n::Int, seed::Int)
        new(
            SimpleGraph(n),       # g
            trues(n),          # active_nodes 
            falses(n),          # informed_nodes 
            MersenneTwister(seed) # rng
        )
    end
end

function ev_raes_step_zero!(state::evraes_graph, lambda::Float64)
    n_old = nv(state.g)
    num_nuovi_nodi = rand(state.rng, Poisson(lambda))
    if num_nuovi_nodi == 0
        return n_old
    end
    
    add_vertices!(state.g, num_nuovi_nodi)

    append!(state.active_nodes, trues(num_nuovi_nodi))
    append!(state.informed_nodes, falses(num_nuovi_nodi)) 
    
    return n_old
end

function ev_raes_step_one!(state::evraes_graph, d::Int, n_old::Int)
    g = state.g
    n_total = nv(g) 
    new_edges = Tuple{Int, Int}[]
    deg_snapshot = degree(g)

    for u in 1:n_total
        if !state.active_nodes[u]
            continue
        end
        
        need = max(0, d - deg_snapshot[u])
        need == 0 && continue

        forbidden = Set(neighbors(g, u))
        push!(forbidden, u)

        count = 0

        attempts = 0
        max_attempts = need * 50 

        while count < need && attempts < max_attempts
            attempts += 1

            target = rand(state.rng, 1:n_old)

            if state.active_nodes[target] && target ∉ forbidden
                push!(new_edges, (u, target))
                #push!(forbidden, target) without replacement
                count += 1
            end
        end
    end
    
    for (u, v) in new_edges
        !has_edge(g, u, v) && add_edge!(g, u, v)
    end
end

function ev_raes_step_two!(state::evraes_graph, d::Int, c::Real)
    g = state.g
    rng = state.rng
    max_deg = ceil(Int, d * c)


    edges_to_remove = Set{Edge{Int}}() 
    

    deg_snapshot = degree(g)


    for u in findall(state.active_nodes)
        
        deg_u = deg_snapshot[u] 
        excess = max(0, deg_u - max_deg)
        
        excess == 0 && continue
        
        current_neighbors = neighbors(g, u)
        
        neighbors_to_drop = sample(rng, current_neighbors, excess, replace=true)
        
        for v in neighbors_to_drop
            push!(edges_to_remove, Edge(u, v)) 
        end
    end
    
    for e in edges_to_remove
        rem_edge!(g, e)
    end
end

function ev_raes_step_three!(state::evraes_graph, p::Real)
    local g = state.g
    p == 0 && return g
        for e in collect(edges(g))
            if rand(state.rng) < p
                rem_edge!(g, src(e), dst(e))
            end
        end

    return g
end

function ev_raes_step_four!(state::evraes_graph, q::Float64)
    g = state.g
    nodes_to_remove = Int[]
    
    for u in findall(state.active_nodes)
        if rand(state.rng) < q
            push!(nodes_to_remove, u)
        end
    end

    if isempty(nodes_to_remove)
        return
    end

    for u in nodes_to_remove
        state.active_nodes[u] = false
        state.informed_nodes[u] = false

        neighbors_copy = collect(neighbors(g, u))
        for v in neighbors_copy
            rem_edge!(g, u, v)
        end
    end
end

ev_raes_step_four! (generic function with 1 method)

In [61]:
function count_active_edges(g::SimpleGraph, active::BitVector)
    s = 0
    for u in vertices(g)
        if active[u]
            for v in neighbors(g, u)
                if active[v] && u < v  
                    s += 1
                end
            end
        end
    end
    return s
end

function run_debug_experiment(params)
    state = evraes_graph(params.n, 123)
    λ = Float64(params.q * params.n)

    

    println("Round | Nodi Attivi | Nodi Post step 0 | Archi Attivi | Nodi < d (Pre 1) | Nodi < d (Post 1) | Archi Post-S1 | Nodi > c*d (Pre 2) | Archi Post-S2 | Nodi < d (Post 2) | Nodi > c*d (Post 2) | Archi Post 3  | Nodi Post 4 | Archi Post 4")
    println("------|-------------|------------------|--------------|------------------|-------------------|---------------|--------------------|---------------|-------------------|---------------------|---------------|-------------|-------------")

    for r in 1:params.rounds
        nodi_attivi = count(state.active_nodes)
        archi_attivi = count_active_edges(state.g, state.active_nodes)

        n_old = ev_raes_step_zero!(state, λ)
        nodi_post_step0 = count(state.active_nodes)

        nodi_sotto_d_pre_s1 = count(<(params.d), [degree(state.g, u) for u in vertices(state.g) if state.active_nodes[u]])
        ev_raes_step_one!(state, params.d, n_old)
        nodi_sotto_d_post_s1 = count(<(params.d), [degree(state.g, u) for u in vertices(state.g) if state.active_nodes[u]])
        nodi_sopra_cd_post_s1 = count(>(params.c*params.d), [degree(state.g, u) for u in vertices(state.g) if state.active_nodes[u]])
        archi_post_s1 = count_active_edges(state.g, state.active_nodes)

        ev_raes_step_two!(state, params.d, params.c)
        archi_post_s2 = count_active_edges(state.g, state.active_nodes)
        nodi_sopra_cd_post_s2 = count(>(params.c*params.d), [degree(state.g, u) for u in vertices(state.g) if state.active_nodes[u]])
        nodi_sotto_d_post_s2 = count(<(params.d), [degree(state.g, u) for u in vertices(state.g) if state.active_nodes[u]])

        ev_raes_step_three!(state, params.p)
        archi_post_s3 = count_active_edges(state.g, state.active_nodes)

        ev_raes_step_four!(state, params.q)
        nodi_post_s4 = count(state.active_nodes)
        archi_post_s4 = count_active_edges(state.g, state.active_nodes)

        @printf("%5d | %11d | %16d | %12d | %16d | %17d | %13d | %18d | %13d | %17d | %19d | %13d | %11d | %12d \n",
                r, nodi_attivi, nodi_post_step0, archi_attivi,
                nodi_sotto_d_pre_s1, nodi_sotto_d_post_s1, archi_post_s1,
                nodi_sopra_cd_post_s1, archi_post_s2, nodi_sotto_d_post_s2,
                nodi_sopra_cd_post_s2, archi_post_s3, nodi_post_s4, archi_post_s4)
    end
end

run_debug_experiment (generic function with 1 method)

In [62]:
PARAMS = (
    n = 2^15,
    d = 4,
    c = 1.5,
    q = 0.1,
    p = 0.1,
    rounds = 30
)

(n = 32768, d = 4, c = 1.5, q = 0.1, p = 0.1, rounds = 30)

In [63]:
run_debug_experiment(PARAMS)

Round | Nodi Attivi | Nodi Post step 0 | Archi Attivi | Nodi < d (Pre 1) | Nodi < d (Post 1) | Archi Post-S1 | Nodi > c*d (Pre 2) | Archi Post-S2 | Nodi < d (Post 2) | Nodi > c*d (Post 2) | Archi Post 3  | Nodi Post 4 | Archi Post 4
------|-------------|------------------|--------------|------------------|-------------------|---------------|--------------------|---------------|-------------------|---------------------|---------------|-------------|-------------
    1 |       32768 |            36063 |            0 |            36063 |                 0 |        144237 |              26725 |         81901 |              7500 |                1765 |         73607 |       32490 |        59725 
    2 |       32490 |            35797 |        59725 |            17740 |                 1 |         95438 |               6133 |         86934 |              2768 |                 385 |         78181 |       32283 |        63703 
    3 |       32283 |            35514 |        63703 |           

# E-RAES POWER LAW

We define a model where, unlike the classic E-raes where all nodes share a single target degree $d,$ each node $u$ gets an individual target degree $d_u,$ assigned according to a three-component mixture model. 
Each node can fall into one of three different categories, determined during initialization:
- Default nodes, with probability $p_{default}.$ If a node falls in this category, it gets assigned the fixed target degree $d.$
- Downscaling nodes, with probability $p_{downscale}.$ If a node falls in this category, it gets a low target degree sampled uniformly at random between a minimum value $min_{d}$ and $d-1.$
- Upscaling nodes, with probability $p_{upscale}.$ If a node falls in this category, it gets a degree computed as $d$ plus an additional value sampled from a Pareto distribution that is roundend as an integer value. 

In [64]:
mutable struct plaw_eraes_graph
    g::SimpleGraph{Int}
    target_degrees::Vector{Int}
    informed_nodes::BitVector  
    rng::AbstractRNG
    
    function plaw_eraes_graph(n::Int, seed::Int, k::Float64, d::Int, prob_default::Float64, prob_downscale::Float64, prob_upscale::Float64, min_d::Int)
        @assert (prob_default + prob_downscale + prob_upscale) == 1
        local_rng = MersenneTwister(seed)
        degs = Vector{Int}(undef, n)
        
        upscale_distr = Pareto(k, 1.0)
        for i in 1:n
            r = rand(local_rng)

            if r < prob_default
                degs[i] = d
            elseif  r < prob_default + prob_downscale 
                degs[i] = rand(local_rng, min_d:(d - 1))
            else
                upscale_val = rand(local_rng, upscale_distr)
                degs[i] = d + round(Int, upscale_val)
            end
        end

        new(
            SimpleGraph(n),       
            degs,          
            falses(n),          
            MersenneTwister(seed) 
        )
    end
end

function plaw_e_raes_step_one!(state::plaw_eraes_graph) 
    local g = state.g
    local rng = state.rng
    n = nv(g)
    new_edges = Tuple{Int, Int}[]
    deg_snapshot = degree(g)

    for u in 1:n
        d_u = state.target_degrees[u]
        need = max(0, d_u - deg_snapshot[u])
        need == 0 && continue

        forbidden = Set(neighbors(g, u))
        push!(forbidden, u)

        if length(forbidden) >= n 
            continue 
        end

        count = 0

        while count < need
            target = rand(rng, 1:n) 
            if target ∉ forbidden
                push!(new_edges, (u, target))
                #push!(forbidden, target)  #(without replacement)
                count += 1
            end

        end
    end

    for (u, v) in new_edges
        if !has_edge(g, u, v) 
            add_edge!(g, u, v)
        end
    end
    return g
end

function plaw_e_raes_step_two!(state::plaw_eraes_graph, c::Real) 
    local g = state.g
    local rng = state.rng
    deg_snapshot = degree(g)
    
    edges_to_remove = Set{Edge{Int}}()
    
    for u in vertices(g)
        d_u = state.target_degrees[u]
        max_deg_u = ceil(Int, d_u * c)
        deg_u = deg_snapshot[u]
        excess = max(0, deg_u - max_deg_u)
        excess == 0 && continue
        
        current_neighbors = neighbors(g, u)
        to_drop = sample(rng, current_neighbors, excess, replace=true)
        
        for v in to_drop
            push!(edges_to_remove, Edge(min(u,v), max(u,v)))
        end
    end
    

    for e in edges_to_remove
        rem_edge!(g, e)
    end
    
    return g
end

function plaw_e_raes_step_three!(state::plaw_eraes_graph, p::Real)
    local g = state.g
    local rng = state.rng
    p == 0 && return g
        for e in collect(edges(g))
            if rand(rng) < p
                rem_edge!(g, src(e), dst(e))
            end
        end

    return g
end

plaw_e_raes_step_three! (generic function with 1 method)

In [65]:
function count_plaw_metrics(state::plaw_eraes_graph, c::Real)
    g = state.g
    targets = state.target_degrees
    n = nv(g)
    
    sotto_d_u = 0
    sopra_cd_u = 0
    
    current_degrees = degree(g) 
    
    for u in 1:n
        deg_u = current_degrees[u]
        d_u = targets[u]
        max_deg_u = ceil(Int, d_u * c)
        
        if deg_u < d_u
            sotto_d_u += 1
        end
        if deg_u > max_deg_u
            sopra_cd_u += 1
        end
    end
    return (sotto_d_u, sopra_cd_u)
end


function run_debug_experiment_plaw(params)
    

    local state = plaw_eraes_graph(
        params.n, 123, params.k, params.d_default, 
        params.prob_default, params.prob_downscale, 
        params.prob_upscale, params.min_d
    )

    local g = state.g
    local c = params.c
    local p = params.p


    println("Distribuzione gradi target generata (primi 20 nodi):")
    println(state.target_degrees[1:20])

    all_target_degrees = values(state.target_degrees)
    d_def = params.d_default
    

    count_down = count(d -> d < d_def, all_target_degrees)
    count_default = count(d -> d == d_def, all_target_degrees)
    count_up = count(d -> d > d_def, all_target_degrees)
    
    @printf("   - Nodi Downscale (< %d): %d (%.1f%%)\n", d_def, count_down, 100 * count_down / params.n)
    @printf("   - Nodi Default   (== %d): %d (%.1f%%)\n", d_def, count_default, 100 * count_default / params.n)
    @printf("   - Nodi Upscale   (> %d): %d (%.1f%%)\n", d_def, count_up, 100 * count_up / params.n)

    println("-"^143)

    println("\nRound | Archi Iniziali | Nodi < d_u | Nodi < d_u | Nodi > c*d_u| Archi Post-S1/2 | Nodi < d_u | Nodi > c*d_u| Connected |Spectral gap |Archi Finali")
    println("      |                | (Pre S1)   | (Post S1)  | (Post S1)   |                 | (Post S2)  | (Post S2)   |           |             |            ")
    println("------|----------------|------------|------------|-------------|-----------------|------------|-------------|-----------|-------------|------------")
    
    for r in 1:params.rounds

        archi_iniziali = ne(g)
        (nodi_sotto_d_pre_s1, _) = count_plaw_metrics(state, c)


        plaw_e_raes_step_one!(state)
        (nodi_sotto_d_post_s1, nodi_sopra_cd_post_s1) = count_plaw_metrics(state, c)
        
        plaw_e_raes_step_two!(state, c)
        

        archi_post_s12 = ne(g)
        (nodi_sotto_d_post_s2, nodi_sopra_cd_post_s2) = count_plaw_metrics(state, c)
        
        connected = is_connected(g)
        gap = spectral_gap(g) 
        

        plaw_e_raes_step_three!(state, p)
        

        archi_finali = ne(g)
        

        @printf("%5d | %14d | %10d | %10d | %11d | %15d | %10d | %11d | %9s | %11.5f | %11d\n", 
                r, archi_iniziali, nodi_sotto_d_pre_s1, 
                nodi_sotto_d_post_s1, nodi_sopra_cd_post_s1, 
                archi_post_s12, 
                nodi_sotto_d_post_s2, nodi_sopra_cd_post_s2, 
                string(connected), gap, archi_finali)
    end
end



run_debug_experiment_plaw (generic function with 1 method)

In [66]:
PARAMS = (
    n = 2^15, 
    d_default = 4,
    c = 3.0,
    p = 0.1,
    rounds = 50,
    k = 2.0,
    prob_default = 0.5,
    prob_downscale = 0.4,
    prob_upscale = 0.1,
    min_d = 2
)


(n = 32768, d_default = 4, c = 3.0, p = 0.1, rounds = 50, k = 2.0, prob_default = 0.5, prob_downscale = 0.4, prob_upscale = 0.1, min_d = 2)

In [67]:

run_debug_experiment_plaw(PARAMS)

Distribuzione gradi target generata (primi 20 nodi):
[4, 2, 4, 4, 3, 2, 2, 3, 7, 2, 4, 4, 3, 6, 2, 3, 4, 5, 2, 2]
   - Nodi Downscale (< 4): 13055 (39.8%)
   - Nodi Default   (== 4): 16406 (50.1%)
   - Nodi Upscale   (> 4): 3307 (10.1%)
-----------------------------------------------------------------------------------------------------------------------------------------------

Round | Archi Iniziali | Nodi < d_u | Nodi < d_u | Nodi > c*d_u| Archi Post-S1/2 | Nodi < d_u | Nodi > c*d_u| Connected |Spectral gap |Archi Finali
      |                | (Pre S1)   | (Post S1)  | (Post S1)   |                 | (Post S2)  | (Post S2)   |           |             |            
------|----------------|------------|------------|-------------|-----------------|------------|-------------|-----------|-------------|------------
    1 |              0 |      32768 |          0 |        2562 |          113309 |        127 |         303 |      true |     0.29399 |      102056
    2 |         102056 |  

# V-RAES POWER LAW

In [68]:
mutable struct plaw_vraes_graph
    g::SimpleGraph{Int}
    active_nodes::BitVector
    informed_nodes::BitVector
    target_degrees::Dict{Int, Int} 
    rng::AbstractRNG
    
    k::Float64
    d_default::Int
    prob_default::Float64
    prob_downscale::Float64
    prob_upscale::Float64
    min_d::Int
    

    function plaw_vraes_graph(
        n::Int, 
        seed::Int, 
        k::Float64, 
        d::Int, 
        prob_default::Float64, 
        prob_downscale::Float64, 
        prob_upscale::Float64, 
        min_d::Int
    )
        
        @assert (prob_default + prob_downscale + prob_upscale) == 1
        local_rng = MersenneTwister(seed)
        

        target_d_dict = Dict{Int, Int}()
        upscale_distr = Pareto(k, 1.0) 

        for i in 1:n
            r = rand(local_rng)
            if r < prob_default
                target_d_dict[i] = d
            elseif r < prob_default + prob_downscale 
                target_d_dict[i] = rand(local_rng, min_d:(d - 1))
            else
                upscale_val = rand(local_rng, upscale_distr)
                target_d_dict[i] = d + round(Int, upscale_val)
            end
        end

        new(
            SimpleGraph(n),       
            trues(n),          
            falses(n),          
            target_d_dict, 
            local_rng,     
            k, d, prob_default, prob_downscale, prob_upscale, min_d
        )
    end
end

function plaw_v_raes_step_zero!(state::plaw_vraes_graph, lambda::Float64)
    n_old = nv(state.g)
    num_nuovi_nodi = rand(state.rng, Poisson(lambda))
    if num_nuovi_nodi == 0
        return n_old
    end
    

    add_vertices!(state.g, num_nuovi_nodi)
    

    upscale_distr = Pareto(state.k, 1.0)
    
    for i in 1:num_nuovi_nodi
        new_node_id = n_old + i
        

        push!(state.active_nodes, true)
        push!(state.informed_nodes, false)
        

        r = rand(state.rng)
        d_default = state.d_default
        
        local new_d_u::Int
        if r < state.prob_default
            new_d_u = d_default
        elseif r < state.prob_default + state.prob_downscale
            new_d_u = rand(state.rng, state.min_d:(d_default - 1))
        else
            upscale_val = rand(state.rng, upscale_distr)
            new_d_u = d_default + round(Int, upscale_val)
        end
        
        state.target_degrees[new_node_id] = new_d_u
    end
    
    return n_old
end

function plaw_v_raes_step_one!(state::plaw_vraes_graph, n_old::Int)
    g = state.g
    n_total = nv(g) 
    new_edges = Tuple{Int, Int}[]
    deg_snapshot = degree(g)

    for u in 1:n_total
        if !state.active_nodes[u]
            continue
        end
        
        d_u = state.target_degrees[u] 
        
        need = max(0, d_u - deg_snapshot[u])
        need == 0 && continue

        forbidden = Set(neighbors(g, u))
        push!(forbidden, u)

        count = 0
        attempts = 0
        max_attempts = need * 50 

        while count < need && attempts < max_attempts
            attempts += 1
            target = rand(state.rng, 1:n_old) 

            if target <= length(state.active_nodes) && state.active_nodes[target] && target ∉ forbidden
                push!(new_edges, (u, target))
                count += 1
            end
        end
    end
    
    for (u, v) in new_edges
        !has_edge(g, u, v) && add_edge!(g, u, v)
    end
end

function plaw_v_raes_step_two!(state::plaw_vraes_graph, c::Real)
    g = state.g
    rng = state.rng
    deg_snapshot = degree(g)
    edges_to_remove = Set{Edge{Int}}() 

    for u in findall(state.active_nodes)
        
        d_u = state.target_degrees[u]
        max_deg_u = ceil(Int, d_u * c)
        
        deg_u = deg_snapshot[u] 
        excess = max(0, deg_u - max_deg_u)
        excess == 0 && continue
        
        current_neighbors = filter(v -> state.active_nodes[v], neighbors(g, u))
        if isempty(current_neighbors) continue end

        neighbors_to_drop = sample(rng, current_neighbors, excess, replace=true)
        
        for v in neighbors_to_drop
            push!(edges_to_remove, Edge(min(u,v), max(u,v))) 
        end
    end
    
    for e in edges_to_remove
        rem_edge!(g, e)
    end
end

function plaw_v_raes_step_three!(state::plaw_vraes_graph, q::Float64)
    g = state.g
    nodes_to_remove = Int[]
    
    for u in findall(state.active_nodes)
        if rand(state.rng) < q
            push!(nodes_to_remove, u)
        end
    end

    if isempty(nodes_to_remove)
        return
    end

    for u in nodes_to_remove
        state.active_nodes[u] = false
        state.informed_nodes[u] = false
        

        delete!(state.target_degrees, u)


        neighbors_copy = collect(neighbors(g, u))
        for v in neighbors_copy
            rem_edge!(g, u, v)
        end
    end
end

plaw_v_raes_step_three! (generic function with 1 method)

In [69]:
function count_active_edges(g::SimpleGraph, active::BitVector)
    s = 0

    for u in findall(active)
        for v in neighbors(g, u)

            if active[v] && u < v 
                s += 1
            end
        end
    end
    return s
end


function count_plaw_metrics_active(state::plaw_vraes_graph, c::Real)
    g = state.g
    targets = state.target_degrees
    
    sotto_d_u = 0
    sopra_cd_u = 0
    
    current_degrees = degree(g)
    
    for u in findall(state.active_nodes)

        if !haskey(targets, u)
            @warn "Nodo attivo $u non ha un grado target!"
            continue
        end
        
        deg_u = current_degrees[u]
        d_u = targets[u]
        max_deg_u = ceil(Int, d_u * c)
        
        if deg_u < d_u
            sotto_d_u += 1
        end
        if deg_u > max_deg_u
            sopra_cd_u += 1
        end
    end
    return (sotto_d_u, sopra_cd_u)
end


function run_debug_experiment_plaw_vraes(params)
    

    local state = plaw_vraes_graph(
        params.n, 123, params.k, params.d_default, 
        params.prob_default, params.prob_downscale, 
        params.prob_upscale, params.min_d
    )
    

    local lambda = (params.n * params.q) 
    #local lambda = (params.n * params.q) / (1.0 - params.q)

    println("Stato iniziale: Nodi=$(count(state.active_nodes)), Lambda=$(round(lambda, digits=2))")
    println("Distribuzione gradi target (primi 10): ", [state.target_degrees[i] for i in 1:10])

    all_target_degrees = values(state.target_degrees)
    d_def = params.d_default
    

    count_down = count(d -> d < d_def, all_target_degrees)
    count_default = count(d -> d == d_def, all_target_degrees)
    count_up = count(d -> d > d_def, all_target_degrees)
    
    @printf("   - Nodi Downscale (< %d): %d (%.1f%%)\n", d_def, count_down, 100 * count_down / params.n)
    @printf("   - Nodi Default   (== %d): %d (%.1f%%)\n", d_def, count_default, 100 * count_default / params.n)
    @printf("   - Nodi Upscale   (> %d): %d (%.1f%%)\n", d_def, count_up, 100 * count_up / params.n)

    println("-"^143) 
    @printf("%-5s | %-9s | %-10s | %-10s | %-10s | %-11s | %-13s | %-12s | %-12s | %-11s | %-9s | %-10s | %-10s\n",
            "Round", "N_Start", "N_Post_S0", "Archi_Start", "Nodi < d_u", "Nodi < d_u", "Archi_Post_S1", "Nodi > c*d_u", "Archi_Post_S2", "Nodi < d_u", "Nodi > c*d_u", "N_End", "Archi_End")
    @printf("%-5s | %-9s | %-10s | %-11s | %-10s | %-11s | %-13s | %-12s | %-12s | %-11s | %-9s | %-10s | %-10s \n",
            "", "(S0)", "(S0)", "(S0)", "(Pre S1)", "(Post S1)", "(S1)", "(Pre S2)", "(S2)", "(Post S2)", "(Post S2)", "(S3)", "(S3)")
    println("-"^130)


    for r in 1:params.rounds

        nodi_attivi = count(state.active_nodes)
        archi_attivi = count_active_edges(state.g, state.active_nodes)
        (nodi_sotto_d_pre_s1, _) = count_plaw_metrics_active(state, params.c)


        n_old = plaw_v_raes_step_zero!(state, lambda)
        nodi_post_step0 = count(state.active_nodes)


        plaw_v_raes_step_one!(state, n_old)
        (nodi_sotto_d_post_s1, nodi_sopra_cd_pre_s2) = count_plaw_metrics_active(state, params.c)
        archi_post_s1 = count_active_edges(state.g, state.active_nodes)


        plaw_v_raes_step_two!(state, params.c)
        archi_post_s2 = count_active_edges(state.g, state.active_nodes)
        (nodi_sotto_d_post_s2, nodi_sopra_cd_post_s2) = count_plaw_metrics_active(state, params.c)


        plaw_v_raes_step_three!(state, params.q)
        nodi_post_s3 = count(state.active_nodes)
        archi_post_s3 = count_active_edges(state.g, state.active_nodes)


        @printf("%-5d | %-9d | %-10d | %-11d | %-10d | %-11d | %-13d | %-12d | %-12d | %-11d | %-9d | %-10d | %-10d\n",
                r, nodi_attivi, nodi_post_step0, archi_attivi,
                nodi_sotto_d_pre_s1, nodi_sotto_d_post_s1, archi_post_s1,
                nodi_sopra_cd_pre_s2, archi_post_s2, nodi_sotto_d_post_s2,
                nodi_sopra_cd_post_s2, nodi_post_s3, archi_post_s3)
                
        if nodi_post_s3 == 0
            println("Rete vuota, simulazione interrotta.")
            break
        end
    end
end


run_debug_experiment_plaw_vraes (generic function with 1 method)

In [70]:
PARAMS = (
    n = 2^15, 
    d_default = 4,
    c = 3.0,
    q = 0.1,    
    rounds = 50,
    k = 2.0,
    prob_default = 0.5,
    prob_downscale = 0.4,
    prob_upscale = 0.1,
    min_d = 2
)

(n = 32768, d_default = 4, c = 3.0, q = 0.1, rounds = 50, k = 2.0, prob_default = 0.5, prob_downscale = 0.4, prob_upscale = 0.1, min_d = 2)

In [71]:
run_debug_experiment_plaw_vraes(PARAMS)

Stato iniziale: Nodi=32768, Lambda=3276.8
Distribuzione gradi target (primi 10): [4, 2, 4, 4, 3, 2, 2, 3, 7, 2]
   - Nodi Downscale (< 4): 13055 (39.8%)
   - Nodi Default   (== 4): 16406 (50.1%)
   - Nodi Upscale   (> 4): 3307 (10.1%)
-----------------------------------------------------------------------------------------------------------------------------------------------
Round | N_Start   | N_Post_S0  | Archi_Start | Nodi < d_u | Nodi < d_u  | Archi_Post_S1 | Nodi > c*d_u | Archi_Post_S2 | Nodi < d_u  | Nodi > c*d_u | N_End      | Archi_End 
      | (S0)      | (S0)       | (S0)        | (Pre S1)   | (Post S1)   | (S1)          | (Pre S2)     | (S2)         | (Post S2)   | (Post S2) | (S3)       | (S3)       
----------------------------------------------------------------------------------------------------------------------------------
1     | 32768     | 36046      | 0           | 32768      | 0           | 129523        | 3378         | 123412       | 529         | 416       |

# EV-RAES POWER LAW

In [ ]:

mutable struct plaw_evraes_graph
    g::SimpleGraph{Int}
    active_nodes::BitVector
    informed_nodes::BitVector
    target_degrees::Dict{Int, Int} 
    rng::AbstractRNG
    
    k::Float64
    d_default::Int
    prob_default::Float64
    prob_downscale::Float64
    prob_upscale::Float64
    min_d::Int
    
    function plaw_evraes_graph(
        n::Int, 
        seed::Int, 
        k::Float64, 
        d::Int, 
        prob_default::Float64, 
        prob_downscale::Float64, 
        prob_upscale::Float64, 
        min_d::Int
    )
        
        @assert (prob_default + prob_downscale + prob_upscale) == 1
        local_rng = MersenneTwister(seed)
        
        target_d_dict = Dict{Int, Int}()
        upscale_distr = Pareto(k, 1.0) 

        for i in 1:n
            r = rand(local_rng)
            if r < prob_default
                target_d_dict[i] = d
            elseif r < prob_default + prob_downscale 
                target_d_dict[i] = rand(local_rng, min_d:(d - 1))
            else
                upscale_val = rand(local_rng, upscale_distr)
                target_d_dict[i] = d + round(Int, upscale_val)
            end
        end

        new(
            SimpleGraph(n),       
            trues(n),          
            falses(n),          
            target_d_dict, 
            local_rng,     
            k, d, prob_default, prob_downscale, prob_upscale, min_d
        )
    end
end


function plaw_ev_raes_step_zero!(state::plaw_evraes_graph, lambda::Float64)
    n_old = nv(state.g)
    num_nuovi_nodi = rand(state.rng, Poisson(lambda))
    if num_nuovi_nodi == 0
        return n_old
    end
    
    add_vertices!(state.g, num_nuovi_nodi)
    
    upscale_distr = Pareto(state.k, 1.0)
    
    for i in 1:num_nuovi_nodi
        new_node_id = n_old + i
        
        push!(state.active_nodes, true)
        push!(state.informed_nodes, false)
        
        r = rand(state.rng)
        d_default = state.d_default
        local new_d_u::Int
        
        if r < state.prob_default
            new_d_u = d_default
        elseif r < state.prob_default + state.prob_downscale
            new_d_u = rand(state.rng, state.min_d:(d_default - 1))
        else
            upscale_val = rand(state.rng, upscale_distr)
            new_d_u = d_default + round(Int, upscale_val)
        end
        
        state.target_degrees[new_node_id] = new_d_u
    end
    
    return n_old
end


function plaw_ev_raes_step_one!(state::plaw_evraes_graph, n_old::Int)
    g = state.g
    n_total = nv(g) 
    new_edges = Tuple{Int, Int}[]
    deg_snapshot = degree(g)

    for u in 1:n_total
        if !state.active_nodes[u]
            continue
        end
        
        if !haskey(state.target_degrees, u) continue end 
        
        d_u = state.target_degrees[u] 
        need = max(0, d_u - deg_snapshot[u])
        need == 0 && continue

        forbidden = Set(neighbors(g, u))
        push!(forbidden, u)

        count = 0
        attempts = 0
        max_attempts = need * 50 

        while count < need && attempts < max_attempts
            attempts += 1
            target = rand(state.rng, 1:n_old) 

            if target <= length(state.active_nodes) && state.active_nodes[target] && target ∉ forbidden
                push!(new_edges, (u, target))
                count += 1
            end
        end
    end
    
    for (u, v) in new_edges
        !has_edge(g, u, v) && add_edge!(g, u, v)
    end
end


function plaw_ev_raes_step_two!(state::plaw_evraes_graph, c::Real)
    g = state.g
    rng = state.rng
    deg_snapshot = degree(g)
    edges_to_remove = Set{Edge{Int}}() 

    for u in findall(state.active_nodes)
        
        if !haskey(state.target_degrees, u) continue end 
        
        d_u = state.target_degrees[u]
        max_deg_u = ceil(Int, d_u * c)
        
        deg_u = deg_snapshot[u] 
        excess = max(0, deg_u - max_deg_u)
        excess == 0 && continue
        
        current_neighbors = filter(v -> state.active_nodes[v], neighbors(g, u))
        if isempty(current_neighbors) continue end

        neighbors_to_drop = sample(rng, current_neighbors, excess, replace=true)
        
        for v in neighbors_to_drop
            push!(edges_to_remove, Edge(min(u,v), max(u,v))) 
        end
    end
    
    for e in edges_to_remove
        rem_edge!(g, e)
    end
end


function plaw_ev_raes_step_three!(state::plaw_evraes_graph, p::Real)
    local g = state.g
    local rng = state.rng 
    
    p == 0 && return g
    

    for e in collect(edges(g))
        if rand(rng) < p
            rem_edge!(g, e)
        end
    end

    return g
end


function plaw_ev_raes_step_four!(state::plaw_evraes_graph, q::Float64)
    g = state.g
    nodes_to_remove = Int[]
    
    for u in findall(state.active_nodes)
        if rand(state.rng) < q
            push!(nodes_to_remove, u)
        end
    end

    if isempty(nodes_to_remove)
        return
    end

    for u in nodes_to_remove
        state.active_nodes[u] = false
        state.informed_nodes[u] = false
        

        delete!(state.target_degrees, u)


        neighbors_copy = collect(neighbors(g, u))
        for v in neighbors_copy
            rem_edge!(g, u, v)
        end
    end
end

plaw_ev_raes_step_four!

In [ ]:

function count_active_edges(g::SimpleGraph, active::BitVector)
    s = 0
    for u in findall(active)
        for v in neighbors(g, u)
            if v <= length(active) && active[v] && u < v 
                s += 1
            end
        end
    end
    return s
end

function count_plaw_metrics_active(state::plaw_evraes_graph, c::Real)
    g = state.g
    targets = state.target_degrees
    sotto_d_u = 0
    sopra_cd_u = 0
    current_degrees = degree(g)
    
    for u in findall(state.active_nodes)
        if !haskey(targets, u)
            @warn "Nodo attivo $u non ha un grado target"
            continue
        end
        
        deg_u = current_degrees[u]
        d_u = targets[u]
        max_deg_u = ceil(Int, d_u * c)
        
        if deg_u < d_u
            sotto_d_u += 1
        end
        if deg_u > max_deg_u
            sopra_cd_u += 1
        end
    end
    return (sotto_d_u, sopra_cd_u)
end



function run_debug_experiment_plaw_evraes(params)
    
    local state = plaw_evraes_graph(
        params.n, 123, params.k, params.d_default, 
        params.prob_default, params.prob_downscale, 
        params.prob_upscale, params.min_d
    )
    
    local lambda = (params.n * params.q) 
    #local lambda = (params.n * params.q) / (1.0 - params.q)

    println("Stato iniziale: Nodi=$(count(state.active_nodes)), Lambda=$(round(lambda, digits=2))")
    
    all_target_degrees = values(state.target_degrees)
    d_def = params.d_default
    count_down = count(d -> d < d_def, all_target_degrees)
    count_default = count(d -> d == d_def, all_target_degrees)
    count_up = count(d -> d > d_def, all_target_degrees)
    @printf("Distribuzione Gradi: Down=%.1f%%, Default=%.1f%%, Up=%.1f%%\n",
            100*count_down/params.n, 100*count_default/params.n, 100*count_up/params.n)
    
    println("-"^160)
    @printf("%-5s | %-9s | %-10s | %-10s | %-10s | %-11s | %-11s | %-12s | %-10s | %-11s | %-9s | %-12s | %-9s | %-10s\n",
            "Round", "N_Start", "N_Post_S0", "Archi_Start", "Nodi < d_u", "Nodi < d_u", "Archi_Post_S1", "Nodi > c*d_u", "Archi_Post_S2", "Nodi < d_u", "Nodi > c*d_u", "Archi_Post_S3", "N_End", "Archi_End")
    @printf("%-5s | %-9s | %-10s | %-10s | %-10s | %-11s | %-11s | %-12s | %-10s | %-11s | %-9s | %-12s | %-9s | %-10s\n",
            "", "(S0)", "(S0)", "(S1)", "(Pre S1)", "(Post S1)", "(S1)", "(Pre S2)", "(S2)", "(Post S2)", "(Post S2)", "(S3)", "(S4)", "(S4)")
    println("-"^160)


    for r in 1:params.rounds
        nodi_attivi = count(state.active_nodes)
        archi_attivi = count_active_edges(state.g, state.active_nodes)
        (nodi_sotto_d_pre_s1, _) = count_plaw_metrics_active(state, params.c)

        n_old = plaw_ev_raes_step_zero!(state, lambda)
        nodi_post_step0 = count(state.active_nodes)

        plaw_ev_raes_step_one!(state, n_old)
        (nodi_sotto_d_post_s1, nodi_sopra_cd_pre_s2) = count_plaw_metrics_active(state, params.c)
        archi_post_s1 = count_active_edges(state.g, state.active_nodes)

        plaw_ev_raes_step_two!(state, params.c)
        archi_post_s2 = count_active_edges(state.g, state.active_nodes)
        (nodi_sotto_d_post_s2, nodi_sopra_cd_post_s2) = count_plaw_metrics_active(state, params.c)

        plaw_ev_raes_step_three!(state, params.p)
        archi_post_s3 = count_active_edges(state.g, state.active_nodes)

        plaw_ev_raes_step_four!(state, params.q)
        nodi_post_s4 = count(state.active_nodes)
        archi_post_s4 = count_active_edges(state.g, state.active_nodes)

        @printf("%-5d | %-9d | %-10d | %-10d | %-10d | %-11d | %-11d | %-12d | %-10d | %-11d | %-9d | %-12d | %-9d | %-10d\n",
                r, nodi_attivi, nodi_post_step0, archi_attivi,
                nodi_sotto_d_pre_s1, nodi_sotto_d_post_s1, archi_post_s1,
                nodi_sopra_cd_pre_s2, archi_post_s2, nodi_sotto_d_post_s2,
                nodi_sopra_cd_post_s2, archi_post_s3, nodi_post_s4, archi_post_s4)
                
        if nodi_post_s4 == 0
            println("Rete vuota, simulazione interrotta.")
            break
        end
    end
end

run_debug_experiment_plaw_evraes (generic function with 1 method)

In [79]:
PARAMS = (
    n = 2^15,  
    d_default = 4,
    c = 3.0,
    p = 0.1,
    q = 0.1,    
    rounds = 50,
    k = 2.0,
    prob_default = 0.5,
    prob_downscale = 0.4,
    prob_upscale = 0.1,
    min_d = 2
)

(n = 32768, d_default = 4, c = 3.0, p = 0.1, q = 0.1, rounds = 50, k = 2.0, prob_default = 0.5, prob_downscale = 0.4, prob_upscale = 0.1, min_d = 2)

In [80]:
run_debug_experiment_plaw_evraes(PARAMS)

Stato iniziale: Nodi=32768, Lambda=3276.8
Distribuzione Gradi: Down=39.8%, Default=50.1%, Up=10.1%
----------------------------------------------------------------------------------------------------------------------------------------------------------------
Round | N_Start   | N_Post_S0  | Archi_Start | Nodi < d_u | Nodi < d_u  | Archi_Post_S1 | Nodi > c*d_u | Archi_Post_S2 | Nodi < d_u  | Nodi > c*d_u | Archi_Post_S3 | N_End     | Archi_End 
      | (S0)      | (S0)       | (S1)       | (Pre S1)   | (Post S1)   | (S1)        | (Pre S2)     | (S2)       | (Post S2)   | (Post S2) | (S3)         | (S4)      | (S4)      
----------------------------------------------------------------------------------------------------------------------------------------------------------------
1     | 32768     | 36046      | 0          | 32768      | 0           | 129523      | 3378         | 123412     | 529         | 416       | 110874       | 32497     | 90099     
2     | 32497     | 35801      |

# Fonti
[1] Dynamic graph models inspired by the Bitcoin network-formation process: https://art.torvergata.it/retrieve/3c9355c1-a6f7-436b-8800-bb977c883086/3571306.3571398.pdf

[2] Comparing Graph Spectra of Adjacency and Laplacian Matrices: https://arxiv.org/pdf/1712.03769

[3] Lecture notes on spectral analysis of Graphs: https://www.cs.cornell.edu/courses/cs6850/2011sp/spectral.pdf